# [Notebook]Unihack26_Matching_Stress
Two source queries per eligible object; five synthetic conditions, same-lineage four-view galleries. Exact-source retrieval is only a proxy when other views may be valid. Explicitly exclude/report objects with fewer than four selection images. Geometry scored on known source only. No calibration/test or serving change. Parcel3D academic-only.

In [ ]:
from pathlib import Path
import sys,json,subprocess
env=Path('/kaggle/temp/kc-lightglue-env')
subprocess.run([sys.executable,'-m','pip','install','--no-deps','--target',str(env),
 'git+https://github.com/cvg/LightGlue.git@5a650c444718caffc2669dc0094cde908f42dd73',
 'kornia==0.8.2','kornia-rs==0.1.9'],check=True)
sys.path.insert(0,str(env))
SOURCES={'ml/__init__.py': '', 'ml/data/__init__.py': '', 'ml/features/__init__.py': '', 'ml/features/geometry.py': '"""Versioned pixel-only geometry features shared by training and inference."""\nfrom __future__ import annotations\n\nimport cv2\nimport numpy as np\n\nVERSION = "sift-pair-features-v2"\n\n\ndef supported_projection(points, homography, inlier_mask):\n    """Reject degenerate RANSAC output before deriving evidence features."""\n    if (homography is None or inlier_mask is None\n            or homography.shape != (3, 3) or not np.isfinite(homography).all()\n            or abs(np.linalg.det(homography)) < 1e-9\n            or inlier_mask.size != len(points)):\n        return None\n    known = inlier_mask.ravel().astype(bool)\n    if known.sum() < 4:\n        return None\n    selected = points[known]\n    denominator = selected @ homography[2, :2] + homography[2, 2]\n    if not np.isfinite(denominator).all() or (np.abs(denominator) < 1e-8).any():\n        return None\n    projected = cv2.perspectiveTransform(selected[:, None], homography)\n    if projected is None or not np.isfinite(projected).all():\n        return None\n    return known, projected[:, 0]\n\n\ndef describe(image, region=None):\n    rgb = np.asarray(image)\n    scale = min(1.0, 768 / max(rgb.shape[:2]))\n    rgb = cv2.resize(rgb, (round(rgb.shape[1]*scale), round(rgb.shape[0]*scale)))\n    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)\n    mask = np.full(gray.shape, 255, np.uint8)\n    if region is not None:\n        mask[:] = 0\n        points = np.rint(np.array(region)/10000*[rgb.shape[1]-1, rgb.shape[0]-1]).astype(np.int32)\n        cv2.fillPoly(mask, [points], 255)\n    kp, descriptors = cv2.SIFT_create(nfeatures=1600).detectAndCompute(gray, mask)\n    selected = mask > 0\n    laplacian = cv2.Laplacian(gray, cv2.CV_64F)\n    return {"keypoints": kp, "descriptors": descriptors, "shape": gray.shape, "mask": mask,\n            "sharpness": float(laplacian[selected].var()) if selected.any() else 0.,\n            "glare": float((gray[selected] > 248).mean()) if selected.any() else 1.}\n\n\ndef scalar_features(query, reference, cosine: float):\n    qd, rd = query["descriptors"], reference["descriptors"]\n    count, ratio, error, overlap, coverage = 0., 0., float("nan"), 0., 0.\n    if qd is not None and rd is not None and min(len(qd), len(rd)) >= 4:\n        good = [pair[0] for pair in cv2.BFMatcher().knnMatch(qd, rd, k=2)\n                if len(pair) == 2 and pair[0].distance < 0.72 * pair[1].distance]\n        unique = {}\n        for match in sorted(good, key=lambda match: match.distance):\n            unique.setdefault(match.trainIdx, match)\n        good = list(unique.values())\n        if len(good) >= 4:\n            qp = np.float32([query["keypoints"][match.queryIdx].pt for match in good])\n            rp = np.float32([reference["keypoints"][match.trainIdx].pt for match in good])\n            cv2.setRNGSeed(26)\n            H, inliers = cv2.findHomography(qp, rp, cv2.RANSAC, 4., maxIters=2000)\n            support = supported_projection(qp, H, inliers)\n            if support is not None:\n                known, projected = support\n                count, ratio = float(known.sum()), float(known.mean())\n                error = float(np.median(np.linalg.norm(projected-rp[known], axis=1))) / 768\n                h, w = query["shape"]\n                valid = cv2.warpPerspective(np.full(reference["shape"], 255, np.uint8), np.linalg.inv(H), (w, h))\n                roi = query["mask"] > 0\n                overlap = float((valid[roi] > 250).mean()) if roi.any() else 0.\n                hull = np.zeros((h, w), np.uint8)\n                if count >= 3:\n                    cv2.fillConvexPoly(hull, cv2.convexHull(qp[known]).astype(np.int32), 1)\n                coverage = float(hull[roi].mean()) if roi.any() else 0.\n    resolution = min(query["shape"]) / max(min(reference["shape"]), 1)\n    return np.array([cosine, min(count/100, 1), ratio, error, overlap, coverage,\n                     resolution, np.log1p(query["sharpness"]), np.log1p(reference["sharpness"]),\n                     query["glare"], reference["glare"], np.nan], dtype=np.float32)\n', 'ml/data/parcel3d_audit.py': '"""CPU-only audit of a mounted Parcel3D input. Does not copy its image archive."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport platform\nfrom collections import Counter, deque\nfrom datetime import UTC, datetime\nfrom pathlib import Path, PurePosixPath\n\nfrom PIL import Image\n\nSOURCE = "ayesha19765/parcel3d"\nORIGINAL = "https://zenodo.org/records/8032204"\nALLOWED_SPLITS = ("train", "validation", "test")\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef within_root(root: Path, relative: str) -> Path:\n    relative = relative.replace("\\\\", "/")\n    parsed = PurePosixPath(relative)\n    if parsed.is_absolute() or ".." in parsed.parts or ":" in relative:\n        raise ValueError("UNSAFE_ANNOTATION_PATH")\n    result = (root / Path(*parsed.parts)).resolve()\n    if not result.is_relative_to(root.resolve()):\n        raise ValueError("PATH_OUTSIDE_INPUT")\n    return result\n\n\ndef find_annotations(input_root: Path) -> dict[str, Path]:\n    # Kaggle can expose both legacy and owner-qualified aliases of one input.\n    # Annotation files sit above per-image directories; never walk 13,400 samples\n    # just to locate three files. Bound discovery and compare duplicate contents.\n    roots, queue, visited = [], deque([(input_root, 0)]), set()\n    while queue:\n        directory, depth = queue.popleft()\n        resolved = directory.resolve()\n        if resolved in visited:\n            continue\n        visited.add(resolved)\n        if len(visited) > 200:\n            raise RuntimeError("ANNOTATION_DISCOVERY_LIMIT")\n        if all((directory / f"parcel3d_{split}.json").is_file() for split in ALLOWED_SPLITS):\n            roots.append(directory)\n            continue\n        if depth < 5:\n            for child in sorted(directory.iterdir()):\n                if (\n                    child.name not in {*ALLOWED_SPLITS, "misc"}\n                    and not child.name.startswith(".")\n                    and child.is_dir()\n                ):\n                    queue.append((child, depth + 1))\n    if not roots:\n        raise RuntimeError("ANNOTATION_ROOT_NOT_FOUND")\n    roots.sort(key=lambda path: (len(path.parts), str(path)))\n    if len(roots) > 1:\n        digests = {\n            tuple(sha256_file(root / f"parcel3d_{split}.json") for split in ALLOWED_SPLITS)\n            for root in roots\n        }\n        if len(digests) != 1:\n            raise RuntimeError("ANNOTATION_ALIASES_HAVE_DIFFERENT_CONTENT")\n    print(f"Annotation roots: {len(roots)} equivalent mount(s); selected {roots[0]}", flush=True)\n    return {split: roots[0] / f"parcel3d_{split}.json" for split in ALLOWED_SPLITS}\n\n\ndef resolve_image(annotation_path: Path, image_record: dict, split: str) -> Path | None:\n    filename = image_record.get("file_name")\n    if not isinstance(filename, str):\n        return None\n    candidates = [\n        within_root(annotation_path.parent, filename),\n        within_root(annotation_path.parent, f"{split}/{filename}"),\n    ]\n    for candidate in candidates:\n        if candidate.is_file():\n            return candidate\n    return None\n\n\ndef audit(input_root: Path, output_root: Path, source_version: int = 1) -> dict:\n    input_root = input_root.resolve()\n    output_root = output_root.resolve()\n    if output_root.is_relative_to(input_root):\n        raise ValueError("OUTPUT_MUST_NOT_BE_INSIDE_READ_ONLY_INPUT")\n    output_root.mkdir(parents=True, exist_ok=True)\n    print("Locating Parcel3D annotation roots (bounded discovery)", flush=True)\n    annotations = find_annotations(input_root)\n    result = {\n        "schema_version": "parcel3d-audit-v2",\n        "source": SOURCE,\n        "source_version": source_version,\n        "original_record": ORIGINAL,\n        "checked_at": datetime.now(UTC).isoformat(),\n        "runtime": {"python": platform.python_version(), "platform": platform.platform()},\n        "source_kind": "SYNTHETIC",\n        "splits": {},\n        "errors": [],\n        "license": {\n            "mirror_metadata": "Apache-2.0; not authoritative for upstream assets",\n            "original_full_dataset": "Academic research only; Zenodo other-nc",\n            "textureless_obj_meshes": "CC-BY-4.0, separate grant in original record",\n            "cardboard_textures": "CC-BY-4.0, separate record 8041823",\n            "training_policy": "Full rendered-image training remains gated pending scope review.",\n            "commercial_release": "NOT_CLEARED_FOR_ORIGINAL_RENDERED_IMAGES",\n            "reference_matching_labels": "Must inspect; COCO damage labels alone do not establish cross-view correspondence or evidence sufficiency.",\n        },\n    }\n    for package in ("numpy", "Pillow", "torch", "opencv-python"):\n        try:\n            result["runtime"][package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            pass\n    group_sets = {}\n    sample_directories = set()\n    for split, path in annotations.items():\n        print(f"Auditing {split}: {path.name}", flush=True)\n        document = json.loads(path.read_text(encoding="utf-8"))\n        images = document.get("images", [])\n        labels = document.get("annotations", [])\n        record = {\n            "annotation_path": str(path.relative_to(input_root)),\n            "annotation_sha256": sha256_file(path),\n            "annotation_bytes": path.stat().st_size,\n            "top_level_keys": sorted(document),\n            "image_count": len(images),\n            "annotation_count": len(labels),\n            "categories": document.get("categories", []),\n            "image_fields": dict(Counter(k for image in images for k in image)),\n            "annotation_fields": dict(Counter(k for annotation in labels for k in annotation)),\n            "sample_records": images[:2],\n            "sample_annotations": labels[:2],\n            "samples": [],\n        }\n        ids = [image.get("id") for image in images]\n        record["duplicate_image_ids"] = len(ids) - len(set(ids))\n        group_key = next(\n            (\n                key\n                for key in ("mesh_id", "object_id", "parcel_id", "model_id")\n                if images and all(key in item for item in images)\n            ),\n            None,\n        )\n        record["explicit_group_key"] = group_key\n        record["group_split_verified"] = False\n        if group_key:\n            group_sets[split] = {str(item[group_key]) for item in images}\n        # Deterministic stratified positions; no exhaustive image read or copy.\n        positions = sorted({int(i * max(len(images) - 1, 0) / 7) for i in range(8)})\n        for index in positions if images else []:\n            image_record = images[index]\n            image_path = resolve_image(path, image_record, split)\n            sample = {\n                "image_id": image_record.get("id"),\n                "file_name": image_record.get("file_name"),\n            }\n            if image_path is None:\n                sample["status"] = "PATH_UNRESOLVED"\n                result["errors"].append(\n                    {"split": split, "image_id": sample["image_id"], "code": "PATH_UNRESOLVED"}\n                )\n            else:\n                try:\n                    with Image.open(image_path) as image:\n                        image.load()\n                        sample.update(\n                            status="READ_OK",\n                            width=image.width,\n                            height=image.height,\n                            format=image.format,\n                            mode=image.mode,\n                        )\n                    sample["sha256"] = sha256_file(image_path)\n                    sample["bytes"] = image_path.stat().st_size\n                    sample["relative_path"] = str(image_path.relative_to(input_root))\n                    sample_directories.add(image_path.parent)\n                except (OSError, ValueError) as exc:\n                    sample.update(status="READ_FAILED", error_type=type(exc).__name__)\n                    result["errors"].append(\n                        {"split": split, "image_id": sample["image_id"], "code": "READ_FAILED"}\n                    )\n            record["samples"].append(sample)\n        result["splits"][split] = record\n    root = annotations["train"].parent\n    extensions = Counter()\n    mesh_examples = []\n    metadata_examples = []\n    # Inventory only sampled image neighborhoods. Do not imply a complete count.\n    print(f"Inspecting metadata/meshes near {len(sample_directories)} sampled images", flush=True)\n    for sample_directory in sorted(sample_directories):\n        for directory, _, filenames in os.walk(sample_directory):\n            for filename in sorted(filenames):\n                suffix = Path(filename).suffix.lower()\n                extensions[suffix] += 1\n                path = Path(directory) / filename\n                if suffix == ".obj" and len(mesh_examples) < 8:\n                    mesh_examples.append(\n                        {\n                            "path": str(path.relative_to(root)),\n                            "bytes": path.stat().st_size,\n                            "sha256": sha256_file(path),\n                        }\n                    )\n                if (\n                    suffix == ".json"\n                    and path not in annotations.values()\n                    and len(metadata_examples) < 8\n                ):\n                    meta = {"path": str(path.relative_to(root)), "bytes": path.stat().st_size}\n                    if path.stat().st_size < 100_000:\n                        try:\n                            meta["document"] = json.loads(path.read_text(encoding="utf-8"))\n                        except (ValueError, UnicodeError):\n                            meta["status"] = "UNREADABLE_JSON"\n                    metadata_examples.append(meta)\n    result["inventory"] = {\n        "scope": "SAMPLED_IMAGE_NEIGHBORHOODS",\n        "sample_directory_count": len(sample_directories),\n        "sampled_extension_counts": dict(extensions),\n        "mesh_examples": mesh_examples,\n        "metadata_examples": metadata_examples,\n    }\n    if len(group_sets) == 3:\n        overlaps = {\n            f"{a}:{b}": len(group_sets[a] & group_sets[b])\n            for a, b in (("train", "validation"), ("train", "test"), ("validation", "test"))\n        }\n        result["group_overlap_counts"] = overlaps\n        for record in result["splits"].values():\n            record["group_split_verified"] = not any(overlaps.values())\n    result["mount_read_gate"] = (\n        "PASS"\n        if not result["errors"]\n        and all(\n            any(s["status"] == "READ_OK" for s in record["samples"])\n            for record in result["splits"].values()\n        )\n        else "FAIL"\n    )\n    result["correspondence_label_gate"] = "PENDING_REVIEW"\n    result["commercial_license_gate"] = "HOLD_ORIGINAL_RENDERS"\n    result["full_archive_copied"] = False\n    target = output_root / "parcel3d-audit.json"\n    target.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")\n    (output_root / "audit-sha256.txt").write_text(\n        sha256_file(target) + "  parcel3d-audit.json\\n", encoding="ascii"\n    )\n    print(\n        json.dumps(\n            {\n                "mount_read_gate": result["mount_read_gate"],\n                "counts": {k: v["image_count"] for k, v in result["splits"].items()},\n                "outputs": str(output_root),\n            },\n            ensure_ascii=False,\n        )\n    )\n    return result\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--input", type=Path, default=Path("/kaggle/input"))\n    parser.add_argument("--output", type=Path, default=Path("/kaggle/working/unihack26-audit"))\n    parser.add_argument("--source-version", type=int, default=1)\n    args = parser.parse_args()\n    audit(args.input, args.output, args.source_version)\n', 'ml/data/parcel_pairs_v2.py': '"""Appearance-change pretraining on audited Parcel3D pixels; not physical dent labels."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\n\nfrom ml.data.parcel3d_audit import within_root\n\nMANIFEST_SHA = "267dd46092d3429dc66f539fb897e79b4be3f68beb94b75cd19f7289250e4d38"\nPAIR_VERSION = "parcel-appearance-pairs-v2"\n\n\ndef appearance_pair(rgb, foreground, seed, scenario):\n    """Aligned pair and exact masks for introduced appearance changes only."""\n    rng = np.random.default_rng(seed)\n    h, w = foreground.shape\n    reference = rgb.copy()\n    region = cv2.erode(foreground.astype(np.uint8), np.ones((7, 7), np.uint8))\n    coordinates = np.argwhere(region > 0)\n    if len(coordinates) < 16:\n        raise ValueError("INSUFFICIENT_FOREGROUND")\n    y, x = coordinates[int(rng.integers(len(coordinates)))]\n    change = np.zeros((h, w), np.uint8)\n    if scenario in (1, 3):\n        end = (int(np.clip(x + rng.integers(-w // 6, w // 6 + 1), 0, w - 1)),\n               int(np.clip(y + rng.integers(-h // 6, h // 6 + 1), 0, h - 1)))\n        cv2.line(change, (int(x), int(y)), end, 1, int(rng.integers(2, 6)))\n    elif scenario in (2, 5):\n        radius = max(4, min(h, w) // 20)\n        pts = np.array([x, y]) + rng.integers(-radius, radius + 1, (8, 2))\n        cv2.fillConvexPoly(change, cv2.convexHull(pts.astype(np.int32)), 1)\n    change *= region\n    query = reference.copy()\n    if change.any():\n        color = rng.integers(20, 240, 3).astype(np.float32)\n        alpha = rng.uniform(.45, .95)\n        query[change > 0] = np.clip(query[change > 0] * (1 - alpha) + color * alpha, 0, 255)\n    if scenario == 3:\n        # A pre-existing mark appears in BOTH images. Its target is unchanged.\n        reference = query.copy()\n        change[:] = 0\n    gradient = np.linspace(rng.uniform(.8, 1.2), rng.uniform(.8, 1.2), w)[None, :, None]\n    query = np.clip(query * gradient + rng.uniform(-10, 10), 0, 255).astype(np.uint8)\n    if rng.random() < .3:\n        query = cv2.GaussianBlur(query, (3, 3), .5)\n    valid = np.ones((h, w), np.uint8)\n    if scenario in (4, 5):\n        cx, cy = int(rng.integers(w // 4, w // 2)), int(rng.integers(h // 4, h // 2))\n        ex, ey = min(w - 1, cx + w // 5), min(h - 1, cy + h // 5)\n        query[cy:ey, cx:ex] = rng.integers(20, 240, 3)\n        valid[max(0, cy - 3):min(h, ey + 3), max(0, cx - 3):min(w, ex + 3)] = 0\n    shift = np.float32([[1, 0, rng.uniform(-.6, .6)], [0, 1, rng.uniform(-.6, .6)]])\n    reference = cv2.warpAffine(reference, shift, (w, h), borderMode=cv2.BORDER_REFLECT_101)\n    valid[:3] = valid[-3:] = 0\n    valid[:, :3] = valid[:, -3:] = 0\n    def convert(a):\n        return torch.from_numpy(np.ascontiguousarray(a.transpose(2, 0, 1))).float() / 255\n    return {"query": convert(query), "reference": convert(reference),\n            "target": torch.from_numpy(change[None].astype(np.float32)),\n            "valid": torch.from_numpy(valid[None].astype(np.float32))}\n\n\nclass ParcelPairs(Dataset):\n    def __init__(self, root, manifest, role, size=392, fixture=False):\n        if role not in ("train", "selection", "calibration", "test") or size % 14:\n            raise ValueError("INVALID_ROLE_OR_RESOLUTION")\n        self.root, self.role, self.size, self.epoch = Path(root), role, size, 0\n        raw = Path(manifest).read_bytes()\n        self.manifest_sha256 = hashlib.sha256(raw).hexdigest()\n        if not fixture and self.manifest_sha256 != MANIFEST_SHA:\n            raise ValueError("AUDITED_MANIFEST_HASH_MISMATCH")\n        self.rows = sorted((json.loads(line) for line in raw.splitlines()), key=lambda r: r["image_path"])\n        self.rows = [r for r in self.rows if r["role"] == role]\n        if not self.rows:\n            raise ValueError("EMPTY_ROLE")\n        self.annotations = {}\n        for split in sorted({r["source_split"] for r in self.rows}):\n            document = json.loads((self.root / f"parcel3d_{split}.json").read_text())\n            images = {item["id"]: item["file_name"] for item in document["images"]}\n            for annotation in document["annotations"]:\n                self.annotations[images[annotation["image_id"]]] = annotation\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, index):\n        row = self.rows[index]\n        raw = within_root(self.root, row["image_path"]).read_bytes()\n        if hashlib.sha256(raw).hexdigest() != row["sha256"]:\n            raise ValueError("IMAGE_CHECKSUM_MISMATCH")\n        import io\n        with Image.open(io.BytesIO(raw)) as source:\n            rgb = np.asarray(source.convert("RGB"))\n        annotation = self.annotations[row["image_path"]]\n        if annotation.get("bbox_mode") != 1 or not isinstance(annotation["segmentation"], list):\n            raise ValueError("UNSUPPORTED_PARCEL_ANNOTATION")\n        foreground = np.zeros(rgb.shape[:2], np.uint8)\n        polygons = [np.array(p, dtype=np.float32).reshape(-1, 2).round().astype(np.int32)\n                    for p in annotation["segmentation"]]\n        cv2.fillPoly(foreground, polygons, 1)\n        x, y, w, h = annotation["bbox"]\n        pad = int(max(w, h) * .12)\n        x0, y0 = max(0, int(x) - pad), max(0, int(y) - pad)\n        x1, y1 = min(rgb.shape[1], int(x + w) + pad), min(rgb.shape[0], int(y + h) + pad)\n        rgb = cv2.resize(rgb[y0:y1, x0:x1], (self.size, self.size))\n        foreground = cv2.resize(foreground[y0:y1, x0:x1], (self.size, self.size), interpolation=cv2.INTER_NEAREST)\n        epoch = self.epoch if self.role == "train" else 0\n        identity = int(hashlib.sha256(row["image_path"].encode()).hexdigest()[:8], 16)\n        scenario = (index + epoch) % 6\n        pair = appearance_pair(rgb, foreground, [26, identity, epoch], scenario)\n        return {**pair, "source": row["image_path"], "lineage": row["lineage_id"], "scenario": scenario}\n', 'ml/learned_matching.py': '"""Optional research matcher. Does not change the serving SIFT implementation."""\nfrom __future__ import annotations\n\nimport hashlib\nimport inspect\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nfrom ml.features.geometry import supported_projection\n\nLIGHTGLUE_REVISION = "5a650c444718caffc2669dc0094cde908f42dd73"\nALIKED_REVISION = "683d7c65197395c0b3f01ebe76e1084a27e73a65"\n\n\ndef fit_alignment(reference_points, query_points, shape):\n    r, q = np.asarray(reference_points, np.float32), np.asarray(query_points, np.float32)\n    empty = {"homography": None, "inliers": 0, "ratio": 0., "coverage": 0., "score": 0.}\n    if r.shape != q.shape or r.ndim != 2 or r.shape[1] != 2:\n        raise ValueError("INVALID_MATCH_COORDINATES")\n    if len(r) < 4 or not np.isfinite(r).all() or not np.isfinite(q).all():\n        return empty\n    cv2.setRNGSeed(26)\n    homography, mask = cv2.findHomography(r, q, cv2.RANSAC, 4., maxIters=2000)\n    support = supported_projection(r, homography, mask)\n    if support is None:\n        return empty\n    known, _ = support\n    h, w = shape[:2]\n    hull = np.zeros((h, w), np.uint8)\n    cv2.fillConvexPoly(hull, cv2.convexHull(q[known]).round().astype(np.int32), 1)\n    coverage = float(hull.mean())\n    count, ratio = int(known.sum()), float(known.mean())\n    return {"homography": homography, "inliers": count, "ratio": ratio,\n            "coverage": coverage, "score": count * ratio * np.sqrt(coverage)}\n\n\nclass SIFTMatcher:\n    def __init__(self):\n        self.extractor = cv2.SIFT_create(nfeatures=1600)\n\n    def extract(self, rgb):\n        keypoints, descriptors = self.extractor.detectAndCompute(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), None)\n        return np.float32([p.pt for p in keypoints]).reshape(-1, 2), descriptors\n\n    def match(self, reference, query):\n        rp, rd = reference\n        qp, qd = query\n        if rd is None or qd is None or min(len(rd), len(qd)) < 4:\n            return np.empty((0, 2), np.float32), np.empty((0, 2), np.float32)\n        good = [p[0] for p in cv2.BFMatcher().knnMatch(rd, qd, k=2)\n                if len(p) == 2 and p[0].distance < .72 * p[1].distance]\n        unique = {}\n        for match in sorted(good, key=lambda p: p.distance):\n            unique.setdefault(match.trainIdx, match)\n        good = list(unique.values())\n        return (np.float32([rp[m.queryIdx] for m in good]).reshape(-1, 2),\n                np.float32([qp[m.trainIdx] for m in good]).reshape(-1, 2))\n\n\nclass AlikedLightGlueMatcher:\n    def __init__(self, cache_dir, device="cuda"):\n        import torch\n        from lightglue import ALIKED, LightGlue\n\n        self.torch, self.device = torch, device\n        if device == "cuda" and not torch.cuda.is_available():\n            raise ValueError("MATCHER_PROFILE_REQUIRES_GPU")\n        expected = {ALIKED: "bf579653b2d1b5b4d0c88cefd87113b4ea8ce1ad094fa07e2db41c9bc47b178e",\n                    LightGlue: "dcb75b9cad1985c5e7a537c512d4318d47fbcc99df75b39a513de60e027c039f"}\n        for module, digest in expected.items():\n            if hashlib.sha256(Path(inspect.getfile(module)).read_bytes()).hexdigest() != digest:\n                raise ValueError("MATCHER_SOURCE_REVISION_MISMATCH")\n        torch.hub.set_dir(str(cache_dir))\n        ALIKED.checkpoint_url = f"https://raw.githubusercontent.com/Shiaoming/ALIKED/{ALIKED_REVISION}/models/{{}}.pth"\n        self.extractor = ALIKED(max_num_keypoints=1600).eval().to(device)\n        self.matcher = LightGlue(features="aliked", depth_confidence=-1, width_confidence=-1).eval().to(device)\n        self.weights = {path.name: hashlib.sha256(path.read_bytes()).hexdigest()\n                        for path in (Path(cache_dir) / "checkpoints").glob("*.pth")}\n\n    def extract(self, rgb):\n        tensor = self.torch.from_numpy(np.ascontiguousarray(rgb.transpose(2, 0, 1))).float().to(self.device) / 255\n        with self.torch.no_grad():\n            return self.extractor.extract(tensor, resize=None)\n\n    def match(self, reference, query):\n        with self.torch.no_grad():\n            matches = self.matcher({"image0": reference, "image1": query})["matches"][0]\n            rp = reference["keypoints"][0][matches[:, 0]].cpu().numpy()\n            qp = query["keypoints"][0][matches[:, 1]].cpu().numpy()\n        return rp, qp\n', 'ml/evaluate_matching.py': '"""Known-transform selection benchmark. No claim about real viewpoint performance."""\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\n\nfrom ml.data.parcel3d_audit import find_annotations\nfrom ml.data.parcel_pairs_v2 import ParcelPairs\nfrom ml.learned_matching import (\n    ALIKED_REVISION,\n    LIGHTGLUE_REVISION,\n    AlikedLightGlueMatcher,\n    SIFTMatcher,\n    fit_alignment,\n)\n\n\ndef known_view(rgb, seed):\n    rng = np.random.default_rng(seed)\n    h, w = rgb.shape[:2]\n    source = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])\n    # One planar synthetic transform, not a physically unseen face of a parcel.\n    inset = source * .78 + np.float32([w, h]) * .11\n    target = inset + rng.uniform(-.08, .08, (4, 2)).astype(np.float32) * [w, h]\n    homography = cv2.getPerspectiveTransform(source, target.astype(np.float32))\n    query = cv2.warpPerspective(rgb, homography, (w, h), borderValue=(127, 127, 127))\n    query = np.clip(query.astype(np.float32) * rng.uniform(.75, 1.25) + rng.uniform(-10, 10), 0, 255).astype(np.uint8)\n    return query, homography\n\n\ndef corner_error(estimated, expected, shape):\n    if estimated is None:\n        return None\n    h, w = shape[:2]\n    corners = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])[:, None]\n    a = cv2.perspectiveTransform(corners, estimated)\n    b = cv2.perspectiveTransform(corners, expected)\n    if a is None or b is None or not np.isfinite(a).all():\n        return None\n    return float(np.linalg.norm(a - b, axis=2).mean())\n\n\ndef run(input_root, manifest, output, full_selection=False):\n    torch.set_num_threads(2)\n    root = find_annotations(Path(input_root))["train"].parent\n    dataset = ParcelPairs(root, manifest, "selection", size=518)\n    groups = defaultdict(list)\n    for index, row in enumerate(dataset.rows):\n        groups[row["lineage_id"]].append(index)\n    # Two source images per lineage. These are independent source objects, not\n    # proof of two independent physical parcels per object.\n    indices = [index for lineage in sorted(groups)\n               for index in (groups[lineage] if full_selection else groups[lineage][:2])]\n    images, lineages, paths = [], [], []\n    for index in indices:\n        sample = dataset[index]\n        rgb = (sample["reference"].numpy().transpose(1, 2, 0) * 255).round().astype(np.uint8)\n        images.append(rgb)\n        lineages.append(sample["lineage"])\n        paths.append(sample["source"])\n    matchers = {"sift": SIFTMatcher(), "aliked_lightglue": AlikedLightGlueMatcher("/kaggle/temp/kc-matcher-cache")}\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    records, summaries = [], {}\n    for name, matcher in matchers.items():\n        started = time.perf_counter()\n        references = [matcher.extract(rgb) for rgb in images]\n        if name != "sift":\n            torch.cuda.synchronize()\n        extraction_seconds = time.perf_counter() - started\n        for index, reference_image in enumerate(images):\n            query, truth = known_view(reference_image, [26, index])\n            rng = np.random.default_rng([99, index])\n            distractors = [i for i in range(len(images)) if lineages[i] != lineages[index]]\n            candidates = [index] + rng.choice(distractors, 3, replace=False).tolist()\n            rng.shuffle(candidates)\n            if name != "sift":\n                torch.cuda.synchronize()\n            started = time.perf_counter()\n            query_features = matcher.extract(query)\n            results = []\n            for candidate in candidates:\n                rp, qp = matcher.match(references[candidate], query_features)\n                fit = fit_alignment(rp, qp, query.shape)\n                error = corner_error(fit["homography"], truth, query.shape) if candidate == index else None\n                results.append({"candidate": candidate, "candidate_source": paths[candidate], "matches": len(rp),\n                                **{key: value for key, value in fit.items() if key != "homography"},\n                                "true_pair_corner_error_px": error})\n            if name != "sift":\n                torch.cuda.synchronize()\n            elapsed = time.perf_counter() - started\n            ordered = sorted(results, key=lambda r: r["score"], reverse=True)\n            # Zero-score ties count as no successful retrieval.\n            top1 = ordered[0]["score"] > 0 and ordered[0]["candidate"] == index\n            positive = next(row for row in results if row["candidate"] == index)\n            row = {"matcher": name, "query_source": paths[index], "lineage": lineages[index],\n                   "top1_correct": top1, "true_pair_corner_error_px": positive["true_pair_corner_error_px"],\n                   "query_and_gallery_seconds": elapsed, "candidates": results}\n            records.append(row)\n        selected = [row for row in records if row["matcher"] == name]\n        errors = [r["true_pair_corner_error_px"] for r in selected]\n        summaries[name] = {"queries": len(selected), "source_lineages": len(groups),\n                           "top1_accuracy": float(np.mean([r["top1_correct"] for r in selected])),\n                           "true_pair_alignment_within_3px_rate": float(np.mean([e is not None and e <= 3 for e in errors])),\n                           "true_pair_alignment_failures": sum(e is None for e in errors),\n                           "p95_query_and_4_references_seconds": float(np.percentile([r["query_and_gallery_seconds"] for r in selected], 95)),\n                           "reference_gallery_extraction_seconds": extraction_seconds}\n    report = {"status": "SYNTHETIC_SELECTION_COMPARISON_COMPLETE", "manifest_sha256": dataset.manifest_sha256,\n              "sampling": "ALL_SELECTION_IMAGES" if full_selection else "FIRST_TWO_PER_LINEAGE",\n              "lightglue_revision": LIGHTGLUE_REVISION, "aliked_weights_revision": ALIKED_REVISION,\n              "weights_sha256": matchers["aliked_lightglue"].weights, "summaries": summaries,\n              "torch": torch.__version__, "gpu": torch.cuda.get_device_name(), "promotion": "HOLD",\n              "test_pixels": "NOT_READ", "calibration": "NOT_RUN", "real_box_validation": "NOT_RUN",\n              "limits": ["Known planar transforms of selection renders, not photographs from new viewpoints.",\n                         "Foreground crop is annotation-provided. No parcel detection or segmentation claim.",\n                         "Four-candidate gallery; no no-reference acceptance calibration.",\n                         "SIFT comparator follows feature settings; not the entire A serving pipeline.",\n                         "Reference gallery is cached; timing excludes loading, network and queue."]}\n    (output / "matching-rows.json").write_text(json.dumps(records, indent=2), encoding="utf-8")\n    (output / "matching-report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    print(json.dumps(report), flush=True)\n    return report\n', 'ml/evaluate_matching_stress.py': '"""Known-transform stress probes with same-lineage gallery distractors.\n\nExact-source retrieval is a proxy: another view of the same source object can\nalso be a valid reference. This benchmark cannot assign real wrong-face labels.\n"""\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\n\nfrom ml.data.parcel3d_audit import find_annotations\nfrom ml.data.parcel_pairs_v2 import ParcelPairs\nfrom ml.evaluate_matching import known_view\nfrom ml.learned_matching import AlikedLightGlueMatcher, SIFTMatcher, fit_alignment\n\nSCENARIOS = ("planar", "close_up", "blur", "shadow", "occlusion")\n\n\ndef gallery_lineages(rows):\n    grouped = defaultdict(list)\n    for index, row in enumerate(rows):\n        grouped[row["lineage_id"]].append(index)\n    eligible = {key: values for key, values in grouped.items() if len(values) >= 4}\n    excluded = {key: len(values) for key, values in grouped.items() if len(values) < 4}\n    if not eligible:\n        raise ValueError("NO_FOUR_VIEW_LINEAGES")\n    return eligible, excluded\n\n\ndef query_variant(rgb, scenario, seed):\n    if scenario not in SCENARIOS:\n        raise ValueError("UNKNOWN_STRESS_SCENARIO")\n    h, w = rgb.shape[:2]\n    rng = np.random.default_rng([26, seed])\n    query, truth = known_view(rgb, [26, seed])\n    ignore = np.zeros((h, w), dtype=np.uint8)\n    if scenario == "close_up":\n        fraction = .55\n        x = float(rng.uniform(.08, .37) * w)\n        y = float(rng.uniform(.08, .37) * h)\n        truth = np.float64([[1 / fraction, 0, -x / fraction], [0, 1 / fraction, -y / fraction], [0, 0, 1]])\n        query = cv2.warpPerspective(rgb, truth, (w, h))\n    elif scenario == "blur":\n        query = cv2.GaussianBlur(query, (13, 13), 3.)\n    elif scenario == "shadow":\n        gain = np.linspace(.35, 1.1, w, dtype=np.float32)[None, :, None]\n        query = np.clip(query * gain, 0, 255).astype(np.uint8)\n    elif scenario == "occlusion":\n        # Known synthetic obstruction; do not score hidden pixels as observable.\n        x0, x1, y0, y1 = int(.3 * w), int(.7 * w), int(.2 * h), int(.8 * h)\n        query[y0:y1, x0:x1] = [160, 130, 110]\n        ignore[y0:y1, x0:x1] = 1\n    visible = cv2.warpPerspective(np.ones((h, w), np.uint8), truth, (w, h), flags=cv2.INTER_NEAREST)\n    visible = ((visible > 0) & (ignore == 0)).astype(np.uint8)\n    return query, truth, visible\n\n\ndef visible_transfer_error(estimated, truth, visible):\n    if estimated is None:\n        return None\n    h, w = visible.shape\n    xx, yy = np.meshgrid(np.linspace(.05 * w, .95 * w, 19), np.linspace(.05 * h, .95 * h, 19))\n    points = np.stack((xx.flatten(), yy.flatten()), axis=1).astype(np.float32)\n    pixels = points.round().astype(int)\n    points = points[visible[pixels[:, 1], pixels[:, 0]] > 0]\n    if len(points) < 4:\n        return None\n    reference = cv2.perspectiveTransform(points[:, None], np.linalg.inv(truth))\n    mapped = cv2.perspectiveTransform(reference, estimated)[:, 0]\n    if not np.isfinite(mapped).all():\n        return None\n    errors = np.linalg.norm(mapped - points, axis=1)\n    return {"points": len(points), "mean_px": float(errors.mean()), "p95_px": float(np.percentile(errors, 95))}\n\n\ndef run(input_root, manifest, output):\n    torch.set_num_threads(2)\n    root = find_annotations(Path(input_root))["train"].parent\n    dataset = ParcelPairs(root, manifest, "selection", size=518)\n    by_lineage, excluded = gallery_lineages(dataset.rows)\n    indices = [i for lineage in sorted(by_lineage) for i in by_lineage[lineage][:4]]\n    images, paths, lineages = [], [], []\n    for index in indices:\n        sample = dataset[index]\n        images.append((sample["reference"].numpy().transpose(1, 2, 0) * 255).round().astype(np.uint8))\n        paths.append(sample["source"])\n        lineages.append(sample["lineage"])\n    queries = [i for lineage in sorted(by_lineage)\n               for i in [j for j, value in enumerate(lineages) if value == lineage][:2]]\n    matchers = {"sift": SIFTMatcher(), "aliked_lightglue": AlikedLightGlueMatcher("/kaggle/temp/kc-stress-cache")}\n    records = []\n    for name, matcher in matchers.items():\n        references = [matcher.extract(rgb) for rgb in images]\n        for index in queries:\n            same = [j for j in range(len(images)) if lineages[j] == lineages[index] and j != index]\n            if len(same) < 3:\n                raise ValueError("NEED_FOUR_VIEWS_PER_LINEAGE")\n            candidates = [index, *same[:3]]\n            np.random.default_rng([99, index]).shuffle(candidates)\n            for scenario in SCENARIOS:\n                query, truth, visible = query_variant(images[index], scenario, index)\n                if name != "sift":\n                    torch.cuda.synchronize()\n                started = time.perf_counter()\n                query_features = matcher.extract(query)\n                results = []\n                for candidate in candidates:\n                    rp, qp = matcher.match(references[candidate], query_features)\n                    fit = fit_alignment(rp, qp, query.shape)\n                    error = visible_transfer_error(fit["homography"], truth, visible) if candidate == index else None\n                    results.append({"candidate_source": paths[candidate], "exact_source": candidate == index,\n                                    **{k: v for k, v in fit.items() if k != "homography"},\n                                    "known_source_visible_transfer": error})\n                if name != "sift":\n                    torch.cuda.synchronize()\n                elapsed = time.perf_counter() - started\n                ordered = sorted(results, key=lambda c: c["score"], reverse=True)\n                exact = ordered[0]["exact_source"] and ordered[0]["score"] > ordered[1]["score"]\n                positive = next(r for r in results if r["exact_source"])\n                error = positive["known_source_visible_transfer"]\n                records.append({"matcher": name, "scenario": scenario, "query_source": paths[index],\n                                "lineage": lineages[index], "exact_source_top1_proxy": bool(exact),\n                                "true_pair_visible_p95_within_3px": error is not None and error["p95_px"] <= 3,\n                                "query_gallery_seconds": elapsed, "candidates": results})\n    summaries = {}\n    for name in matchers:\n        summaries[name] = {}\n        for scenario in SCENARIOS:\n            rows = [r for r in records if r["matcher"] == name and r["scenario"] == scenario]\n            summaries[name][scenario] = {"queries": len(rows),\n                "exact_source_top1_proxy": sum(r["exact_source_top1_proxy"] for r in rows),\n                "true_pair_visible_p95_within_3px": sum(r["true_pair_visible_p95_within_3px"] for r in rows),\n                "p95_query_gallery_seconds": float(np.percentile([r["query_gallery_seconds"] for r in rows], 95))}\n    report = {"status": "STRESS_SELECTION_COMPLETE_NOT_REAL_VIEWPOINT_VALIDATION", "summaries": summaries,\n              "source_lineages": len(by_lineage), "source_query_images": len(queries), "scenarios": SCENARIOS,\n              "excluded_lineages_with_fewer_than_four_images": excluded,\n              "manifest_sha256": dataset.manifest_sha256, "weights_sha256": matchers["aliked_lightglue"].weights,\n              "test_pixels": "NOT_READ", "calibration": "NOT_RUN", "promotion": "HOLD",\n              "gpu": torch.cuda.get_device_name(), "torch": torch.__version__,\n              "limits": ["Query transforms, crops, shadows and obstruction are synthetic.",\n                         "Same-lineage gallery may contain multiple valid references; exact-source rank is only a proxy.",\n                         "Geometry is scored only against the known original source, not unknown cross-view correspondence.",\n                         "Provided foreground crop and known visibility are evaluation conveniences, not product capabilities.",\n                         "Two source queries per eligible source object are not independent physical parcels.",\n                         "Objects with fewer than four selection images are explicitly excluded from this four-view probe.",\n                         "Visible-grid p95 transfer differs from full-frame mean corner error; do not compare them as the same metric."]}\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    (output / "stress-rows.json").write_text(json.dumps(records, indent=2), encoding="utf-8")\n    (output / "stress-report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    print(json.dumps(report), flush=True)\n    return report\n'}
root=Path('/kaggle/working/_src')
for name,source in SOURCES.items():
 p=root/name
 p.parent.mkdir(parents=True,exist_ok=True)
 p.write_text(source,encoding='utf-8')
sys.path.insert(0,str(root))
Path('/kaggle/working/source-hashes.json').write_text(json.dumps({'ml/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/data/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/features/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'ml/features/geometry.py': 'c08b867e13ca331207029af95a64959bf8bd7ab3cf8ebbad22290a373347b00b', 'ml/data/parcel3d_audit.py': '08ddf5f683f5b23c8364228d53f8b3883518465c3e2d66a64bf3ff9888f46b4e', 'ml/data/parcel_pairs_v2.py': '26d32013957f1bc2cc281bcf4b8d8a4ba15a702899125313b58b2a5d11f30730', 'ml/learned_matching.py': 'c62c88c83325e6521ce0476ba55b1e7923ac533593fca7cc08fb0ba27a4a17f4', 'ml/evaluate_matching.py': '57fa2ffd083d666efb253b58a2a11ad5422c6bbb92d1013acd4c4564e7ca3a54', 'ml/evaluate_matching_stress.py': 'd0283a60d7052bce9a87e77c661ad3f05817a4d6fedbee8967a128b616fa91c5'},indent=2))


In [ ]:
FULL_SELECTION=True
import hashlib
from collections import deque
from ml.data.parcel_pairs_v2 import MANIFEST_SHA
from ml.evaluate_matching_stress import run
queue=deque([(Path('/kaggle/input'),0)])
manifests=set()
while queue:
    directory,depth=queue.popleft()
    path=directory/'images.jsonl'
    if path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest()==MANIFEST_SHA:
        manifests.add(path.resolve())
    if depth<6:
        for child in directory.iterdir():
            if child.is_dir() and child.name not in {'parcel3d','train','validation','test','misc'}:
                queue.append((child,depth+1))
assert manifests,'AUDITED_MANIFEST_MISSING'
run('/kaggle/input',sorted(manifests)[0],'/kaggle/working/stress')
